# NB1 — Delta Lake Basics (lightweight path)

**Stack:** `deltalake` (delta-rs) + Polars + DuckDB. No Spark, no JVM.
Maps to slide §2 (Delta Lake) + deliverable bullet 1.

> Spark equivalent: `spark.read.format("delta").load(path)` ↔ `DeltaTable(path).to_pyarrow_table()`.
> Same on-disk format, different binding.

**Bản nộp đã thực thi** · Nguyen Truong Bao · MSSV 2A202602540 · K4-Track02-Day18  
Đường chạy: **lightweight** · Python 3.11.9 · Windows 11 Pro · deltalake 1.6.6 · pyiceberg 0.12.0 · duckdb 1.5.6 · polars 1.44.2 · pyarrow 25.0.1  
Thực thi bằng Jupyter kernel (`nbconvert --execute`) ngày 05/10/2026. Các ô *Giải thích kết quả* được thêm sau khi chạy, không thay đổi code hay output.

In [1]:
import _setup  # noqa: F401  -- adds scripts/ to sys.path (file-relative)
import polars as pl
from deltalake import DeltaTable, write_deltalake
from lakehouse import path, reset

table_path = path("scratch", "users_delta")
reset(table_path)  # idempotent rerun

## 1. Write a Delta table

In [2]:
df = pl.DataFrame({
    "id": [1, 2, 3],
    "name": ["alice", "bob", "charlie"],
    "age": [30, 25, 35],
    "city": ["Hanoi", "HCMC", "Danang"],
})
write_deltalake(table_path, df.to_arrow(), mode="overwrite")

## 2. Read it back + inspect transaction log

Look at `_lakehouse/scratch/users_delta/_delta_log/00000000000000000000.json` —
that's the transaction log. Same JSON format Spark/Databricks would write.

In [3]:
dt = DeltaTable(table_path)
print(pl.from_arrow(dt.to_pyarrow_table()))
print("\nHistory:")
for h in dt.history():
    print(f"  v{h['version']}  {h['operation']}  {h.get('operationMetrics', {})}")

shape: (3, 4)
┌─────┬─────────┬─────┬────────┐
│ id  ┆ name    ┆ age ┆ city   │
│ --- ┆ ---     ┆ --- ┆ ---    │
│ i64 ┆ str     ┆ i64 ┆ str    │
╞═════╪═════════╪═════╪════════╡
│ 1   ┆ alice   ┆ 30  ┆ Hanoi  │
│ 2   ┆ bob     ┆ 25  ┆ HCMC   │
│ 3   ┆ charlie ┆ 35  ┆ Danang │
└─────┴─────────┴─────┴────────┘

History:
  v0  WRITE  {'num_added_files': 1, 'num_removed_files': 0, 'num_partitions': 0, 'num_added_rows': 3, 'execution_time_ms': 4, 'num_retries': 0}


## 3. Schema enforcement — try to write a wrong schema

In [4]:
bad = pl.DataFrame({"id": [4], "name": ["dan"], "age": ["thirty"], "city": ["Hue"]})
version_before_bad = DeltaTable(table_path).version()
try:
    write_deltalake(table_path, bad.to_arrow(), mode="append")
    bad_write_blocked = False
    print("UNEXPECTED: bad write succeeded — schema enforcement broken")
except Exception as e:
    bad_write_blocked = True
    msg = str(e).splitlines()[0][:120]
    print(f"BLOCKED by schema enforcement (expected): {type(e).__name__}: {msg}")
# A blocked write must not leave a commit behind: the version stays the same.
bad_write_blocked = bad_write_blocked and DeltaTable(table_path).version() == version_before_bad
print(f"Table version before/after the bad write: {version_before_bad} → {DeltaTable(table_path).version()}")

BLOCKED by schema enforcement (expected): Exception: Cast error: Cannot cast string 'thirty' to value of Int64 type
Table version before/after the bad write: 0 → 0


> **Giải thích kết quả —** Ghi `age='thirty'` (string) vào cột `age` kiểu `long` bị chặn: `Cast error: Cannot cast string 'thirty' to value of Int64 type`. Version bảng **0 → 0**: write thất bại không để lại commit nào (ô 6 bên dưới chỉ có commit 0 và 1, commit 1 là lần evolution hợp lệ). Cơ chế: schema của bảng nằm trong action `metaData` của `_delta_log/`; writer đối chiếu/cast batch mới với schema đó **trước** khi ghi commit JSON, nên dữ liệu sai kiểu không bao giờ trở thành một phần của bảng.

## 4. Schema evolution (opt-in)

In [5]:
new = pl.DataFrame({
    "id": [4], "name": ["dan"], "age": [28], "city": ["Hue"], "tier": ["premium"],
})
write_deltalake(table_path, new.to_arrow(), mode="append", schema_mode="merge")
dt = DeltaTable(table_path)
# Sort by id so the printout is stable across reruns — Delta does not
# preserve write-order across appends.
print(pl.from_arrow(dt.to_pyarrow_table()).sort("id"))

shape: (4, 5)
┌─────┬─────────┬─────┬────────┬─────────┐
│ id  ┆ name    ┆ age ┆ city   ┆ tier    │
│ --- ┆ ---     ┆ --- ┆ ---    ┆ ---     │
│ i64 ┆ str     ┆ i64 ┆ str    ┆ str     │
╞═════╪═════════╪═════╪════════╪═════════╡
│ 1   ┆ alice   ┆ 30  ┆ Hanoi  ┆ null    │
│ 2   ┆ bob     ┆ 25  ┆ HCMC   ┆ null    │
│ 3   ┆ charlie ┆ 35  ┆ Danang ┆ null    │
│ 4   ┆ dan     ┆ 28  ┆ Hue    ┆ premium │
└─────┴─────────┴─────┴────────┴─────────┘


> **Giải thích kết quả —** `schema_mode="merge"` thêm cột `tier` (opt-in): 3 dòng cũ đọc ra `tier = null`, dòng mới là `premium`. Không file cũ nào bị ghi lại — commit 1 chỉ gồm `metaData` mới (schema 5 cột) + `add` một file mới (xem ô 6). Evolution phải bật rõ ràng: kiểm tra riêng cho thấy cùng kiểu append thêm cột nhưng **không** có `schema_mode` bị từ chối với `SchemaMismatchError`, nên đổi schema là quyết định có chủ đích, không phải tác dụng phụ của một job ghi lỗi.

## 5. Query with DuckDB via Arrow (part of the required notebook)

In [6]:
import duckdb

# We hand DuckDB an Arrow table rather than calling `delta_scan()`. delta_scan
# autoloads a DuckDB extension over the network — fine at home, a support
# ticket in a firewalled classroom. Arrow registration is zero-copy and offline.
con = duckdb.connect()
con.register("users", DeltaTable(table_path).to_pyarrow_table())
tier_counts = con.sql("SELECT tier, count(*) AS n FROM users GROUP BY 1 ORDER BY 1").fetchall()
print(tier_counts)

[('premium', 1), (None, 3)]


> **Giải thích kết quả —** DuckDB đọc bảng qua Arrow (không cần tải extension `delta_scan`) và thấy đúng **2 nhóm tier**: `premium` (1 dòng) và `NULL` (3 dòng cũ) — NULL là một nhóm riêng trong `GROUP BY`.

## 6. Evidence: the `_delta_log/` commits on disk

Added for the submission: list the JSON commits and show what each one
records. v0 carries `protocol` + `metaData` (schema) + `add`; the schema-merge
commit carries a new `metaData` whose schema now includes `tier`.

In [7]:
import json  # noqa: E402
from pathlib import Path as _P  # noqa: E402

log_dir = _P(table_path) / "_delta_log"
for f in sorted(log_dir.glob("*.json")):
    actions = [json.loads(line) for line in f.read_text().splitlines() if line.strip()]
    print(f"{f.name}  ({f.stat().st_size} B)  actions={[next(iter(a)) for a in actions]}")

print("\n--- 00000000000000000000.json (first commit) ---")
for line in (log_dir / "00000000000000000000.json").read_text().splitlines():
    a = json.loads(line)
    kind = next(iter(a))
    if kind == "metaData":
        fields = json.loads(a["metaData"]["schemaString"])["fields"]
        print("metaData.schema:", [(fl["name"], fl["type"]) for fl in fields])
    elif kind == "add":
        print("add:", {k: a["add"][k] for k in ("path", "size", "dataChange")})
        print("add.stats:", a["add"]["stats"])
    else:
        print(f"{kind}:", a[kind])

last_commit = sorted(log_dir.glob("*.json"))[-1]
for line in last_commit.read_text().splitlines():
    a = json.loads(line)
    if "metaData" in a:
        fields = json.loads(a["metaData"]["schemaString"])["fields"]
        print(f"\n{last_commit.name} metaData.schema:", [(fl["name"], fl["type"]) for fl in fields])

00000000000000000000.json  (1409 B)  actions=['commitInfo', 'protocol', 'metaData', 'add']
00000000000000000001.json  (1465 B)  actions=['commitInfo', 'metaData', 'add']

--- 00000000000000000000.json (first commit) ---
commitInfo: {'timestamp': 1791168582323, 'operation': 'WRITE', 'operationParameters': {'mode': 'Overwrite'}, 'engineInfo': 'delta-rs:py-1.6.6', 'clientVersion': 'delta-rs.py-1.6.6', 'operationMetrics': {'num_added_files': 1, 'num_removed_files': 0, 'num_partitions': 0, 'num_added_rows': 3, 'execution_time_ms': 4, 'num_retries': 0}}
protocol: {'minReaderVersion': 1, 'minWriterVersion': 2}
metaData.schema: [('id', 'long'), ('name', 'string'), ('age', 'long'), ('city', 'string')]
add: {'path': 'part-00000-ab44feed-e1e2-47cf-9f80-44bcb86aef90-c000.snappy.parquet', 'size': 1384, 'dataChange': True}
add.stats: {"numRecords":3,"minValues":{"id":1,"name":"alice","city":"Danang","age":25},"maxValues":{"city":"Hanoi","age":35,"id":3,"name":"charlie"},"nullCount":{"city":0,"id":0,

> **Giải thích kết quả —** Bằng chứng `_delta_log/`: đúng **2 commit JSON** (`…000.json`, `…001.json`); lần ghi sai kiểu không tạo commit. Commit 0 có 4 action: `commitInfo` (WRITE, engine delta-rs 1.6.6, metrics), `protocol` (minReader 1 / minWriter 2), `metaData` (schema 4 cột) và `add` (đường dẫn file Parquet + `stats` min/max/nullCount). Commit 1 mang `metaData` mới có thêm `tier`. Trạng thái bảng = replay các action này theo thứ tự; `stats` trong `add` chính là thứ NB2 dùng để bỏ qua file.

## ✅ Deliverable check
- [ ] `_delta_log/` contains JSON files
- [ ] Schema enforcement blocked the bad write
- [ ] schema_mode="merge" added the `tier` column
- [ ] DuckDB query returned 2 tier groups
The schema-enforcement flag is set by the bad-write cell: True only if the
write raised AND the table version did not move.

In [8]:
from pathlib import Path as _Path  # noqa: E402

_log = sorted(_Path(table_path).glob("_delta_log/*.json"))
_cols = DeltaTable(table_path).schema().to_arrow().names
checks = {
    "_delta_log/ has JSON commits": len(_log) >= 2,
    "schema enforcement blocked bad write": bad_write_blocked,
    "tier column added via schema_mode=merge": "tier" in _cols,
    "duckdb sees 2 tier groups": len(tier_counts) == 2,
}
for k, v in checks.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
assert all(checks.values()), "NB1 incomplete — see FAIL rows above"
print("\nNB1 complete.")

  [PASS] _delta_log/ has JSON commits
  [PASS] schema enforcement blocked bad write
  [PASS] tier column added via schema_mode=merge
  [PASS] duckdb sees 2 tier groups

NB1 complete.


> **Giải thích kết quả —** Cờ *schema enforcement blocked bad write* không còn hardcode `True` như bản gốc: nó chỉ `True` khi write sai kiểu ném lỗi **và** version bảng không đổi (ô mục 3).